In [1]:
import pandas as pd
from pathlib import Path
import util

pd.set_option('display.float_format', '{:,.0%}'.format)


In [2]:


# list of equity geographies
equity_geogs = util.summary_config['hh_equity_geogs']
# not_equity_geogs = ["NOT in " + item for item in equity_geogs]

## Household VMT

In [3]:
# vmt data
df_vmt = pd.read_csv(util.output_path / 'agg/dash/person_vmt.csv')

# add home RGC
df_vmt['is_rgc'] = 'Not in RGC'
df_vmt.loc[df_vmt['hh_rgc'] != 'Not in RGC', 'is_rgc'] = 'In RGC'

# Select only drivers (dorp = 1) and auto trips
df_vmt = df_vmt[df_vmt['mode'].isin(['SOV','HOV2','HOV3+']) & (df_vmt['dorp'] == 1)].copy()

df_hh = pd.read_csv(util.output_path / 'agg/dash/hh_geog.csv')
df_hh['is_rgc'] = 'Not in RGC'
df_hh.loc[df_hh['hh_rgc'] != 'Not in RGC', 'is_rgc'] = 'In RGC'

def vmt_per_hh(df_vmt, df_hh, geog, map=False):
    _df_vmt = df_vmt.groupby(geog).sum()[['travdist_wt']]
    df_hh = df_hh.groupby(geog).sum()[['hhexpfac']]

    df = _df_vmt.merge(df_hh, left_index=True, right_index=True)

    if map:
        df.index = df.index.astype('int').map({
                                0: 'Below Regional Average', 
                                1: 'Above Regional Average', 
                                2: 'Higher Share of Equity Population',
                                })
    
    
    df.loc['Region',:] = df.sum(axis=0)
    df['Average Miles per Household'] = df['travdist_wt']/df['hhexpfac']

    
    return df[['Average Miles per Household']]


In [4]:
pd.set_option('display.float_format', '{:,.1f}'.format)
vmt_per_hh(df_vmt, df_hh, "hh_county")

,Average Miles per Household
hh_county,
King,32.5
Kitsap,35.6
Outside Region,55.8
Pierce,41.1
Snohomish,44.3
Region,36.6


In [5]:
vmt_per_hh(df_vmt, df_hh, "is_rgc")

,Average Miles per Household
is_rgc,
In RGC,10.8
Not in RGC,39.3
Region,36.6


In [6]:
vmt_per_hh(df_vmt, df_hh, "hh_rgc")

,Average Miles per Household
hh_rgc,
Auburn,25.6
Bellevue,12.3
Bothell Canyon Park,35.9
Bremerton,13.6
Burien,25.3
Everett,16.0
Federal Way,26.2
Greater Downtown Kirkland,22.7
Kent,23.1


In [7]:
pd.set_option('display.float_format', '{:,.1f}'.format)
vmt_per_hh(df_vmt, df_hh, "hh_rg_proposed")

,Average Miles per Household
hh_rg_proposed,
Cities and Towns,50.7
Core Cities,36.4
High Capacity Transit Communities,39.7
Metropolitan Cities,22.0
Rural Areas,61.4
Urban Unincorporated Areas,47.5
Region,36.6


In [8]:
efa_names = {
    "People of Color": "hh_efa_poc",
    "Income": "hh_efa_pov200",
    "LEP": "hh_efa_lep",
    "Disability": "hh_efa_dis",
    "Older Adults": "hh_efa_older",
    "Youth": "hh_efa_youth"
}

df = pd.DataFrame()
for name, col in efa_names.items():
    df[name] = vmt_per_hh(df_vmt, df_hh, col, map=True)
df

,People of Color,Income,LEP,Disability,Older Adults,Youth
hh_efa_poc,,,,,,
Below Regional Average,39.3,39.3,37.0,37.8,35.2,30.0
Above Regional Average,33.4,34.6,36.7,36.7,38.6,43.0
Higher Share of Equity Population,33.2,29.0,34.9,32.0,37.2,48.7
Region,36.6,36.6,36.6,36.6,36.6,36.6


## Delay

In [9]:
df = pd.read_csv(util.output_path / 'agg/dash/trip_time_total.csv')
df = df[(df['mode'].isin(['SOV','HOV2','HOV3+'])&(df['dorp']==1))]

In [10]:
pd.options.display.float_format = '{:0,.1f}'.format
# Hours of delay for households in these locations
# df[['Total Delay Hours']]

def delay_per_hh(geog, map=False):

    df_hh = pd.read_csv(util.output_path / 'agg/dash/hh_geog.csv')
    df_hh['is_rgc'] = 'Not in RGC'
    df_hh.loc[df_hh['hh_rgc'] != 'Not in RGC', 'is_rgc'] = 'In RGC'

    df = pd.read_csv(util.output_path / 'agg/dash/trip_time_total.csv')
    df['is_rgc'] = 'Not in RGC'
    df.loc[df['hh_rgc'] != 'Not in RGC', 'is_rgc'] = 'In RGC'
    df = df[(df['mode'].isin(['SOV','HOV2','HOV3+'])&(df['dorp']==1))]
    df = df.groupby(geog).sum()[['travtime_wt']]

    df2 = pd.read_csv(util.output_path / 'agg/dash/trip_sov_ff_time.csv')
    df2['is_rgc'] = 'Not in RGC'
    df2.loc[df2['hh_rgc'] != 'Not in RGC', 'is_rgc'] = 'In RGC'
    df2 = df2[(df2['mode'].isin(['SOV','HOV2','HOV3+'])&(df2['dorp']==1))]
    df2 = df2.groupby(geog).sum()[['sov_ff_time_wt']]
    df = df2.merge(df, on=geog)

    # Hours of delay from travel time
    df['Total Delay Hours'] = (df['travtime_wt'] - df['sov_ff_time_wt'])/60
    # Set any negative delay to 0
    df.loc[df['Total Delay Hours'] < 0, 'Total Delay Hours'] = 0

    df_hh = df_hh.groupby(geog).sum()[['hhexpfac']]
    
    
    # df.loc['Region',:] = df.sum(axis=0)
    # df['Average Miles per Household'] = df['travdist_wt']/df['hhexpfac']


    df = df.merge(df_hh, left_index=True, right_index=True)
    
    df['Average Minutes of Delay per HH'] = df['Total Delay Hours']/df['hhexpfac']*60

    if map:
        df.index = df.index.astype('int').map({
                                0: 'Below Regional Average', 
                                1: 'Above Regional Average', 
                                2: 'Higher Share of Equity Population',
                                })

    df['Annual Hours of Delay per HH'] = df['Average Minutes of Delay per HH']*util.summary_config['weekday_to_annual']/60

    df[['Total Delay Hours',
        'Annual Hours of Delay per HH']] = df[['Total Delay Hours', 'Annual Hours of Delay per HH']].astype(int).applymap('{:,}'.format)


    return df[['Total Delay Hours','Average Minutes of Delay per HH','Annual Hours of Delay per HH']]

df = delay_per_hh('hh_county')
df

,Total Delay Hours,Average Minutes of Delay per HH,Annual Hours of Delay per HH
hh_county,,,
King,"116,635",7.3,38
Kitsap,"5,829",3.2,17
Outside Region,0,0.0,0
Pierce,"43,363",7.4,39
Snohomish,"51,059",9.6,51


In [11]:
delay_per_hh('is_rgc')


,Total Delay Hours,Average Minutes of Delay per HH,Annual Hours of Delay per HH
is_rgc,,,
In RGC,"6,094",2.2,11
Not in RGC,"210,793",8.1,42


In [12]:
delay_per_hh('hh_rgc')


,Total Delay Hours,Average Minutes of Delay per HH,Annual Hours of Delay per HH
hh_rgc,,,
Auburn,74,4.5,23
Bellevue,519,3.2,16
Bothell Canyon Park,62,13.2,70
Bremerton,71,2.6,13
Burien,162,4.9,26
Everett,164,2.6,13
Federal Way,15,4.1,21
Greater Downtown Kirkland,516,7.0,37
Kent,83,5.2,27


In [13]:
delay_per_hh('hh_rg_proposed')


,Total Delay Hours,Average Minutes of Delay per HH,Annual Hours of Delay per HH
hh_rg_proposed,,,
Cities and Towns,"21,044",9.3,49
Core Cities,"54,742",8.5,45
High Capacity Transit Communities,"56,932",9.5,50
Metropolitan Cities,"46,438",4.7,25
Rural Areas,"27,762",8.3,44
Urban Unincorporated Areas,"9,966",9.7,51


In [14]:
df = pd.DataFrame()
for name, col in efa_names.items():
    _df = delay_per_hh(col, map=True)
    _df['Group'] = name
    df = pd.concat([df, _df])

df = df.reset_index()
df.rename(columns={'index':'EFA Type'}, inplace=True)

df[['Group', 'EFA Type', 'Total Delay Hours', 'Average Minutes of Delay per HH', 'Annual Hours of Delay per HH']]

,Group,EFA Type,Total Delay Hours,Average Minutes of Delay per HH,Annual Hours of Delay per HH
0,People of Color,Below Regional Average,"114,784",7.3,38
1,People of Color,Above Regional Average,"64,894",7.8,41
2,People of Color,Higher Share of Equity Population,"37,208",7.6,40
3,Income,Below Regional Average,"141,861",8.2,43
4,Income,Above Regional Average,"51,030",6.9,36
5,Income,Higher Share of Equity Population,"23,996",5.8,30
6,LEP,Below Regional Average,"124,922",6.9,36
7,LEP,Above Regional Average,"52,055",8.8,46
8,LEP,Higher Share of Equity Population,"39,910",8.4,44
9,Disability,Below Regional Average,"131,743",8.4,44


## Vehicle Ownership Distribution

percentage of households by number of vehicles available, by geography


In [15]:
pd.set_option('display.float_format', '{:,.0%}'.format)

hh = pd.read_csv(util.output_path / 'agg/dash/auto_ownership_efa.csv')
df_hh = hh.copy()
df_hh = df_hh[df_hh['hh_county']!="Outside Region"].copy()
# add home RGC
df_hh['is_rgc'] = 'Not in RGC'
df_hh.loc[df_hh['hh_rgc'] != 'Not in RGC', 'is_rgc'] = 'In RGC'

equity_geogs = ['hh_efa_dis', 'hh_efa_older', 'hh_efa_lep', 'hh_efa_pov200', 'hh_efa_poc', 'hh_efa_youth']
df_hh[equity_geogs] = df_hh[equity_geogs].apply(lambda x: x.\
        map({0: 'Below Regional Average', 
             1: 'Above Regional Average', 
             2: 'Higher Share of Equity Population'}))

df_hh['hhvehs'] = df_hh['hhvehs'].map({0:"0 vehicle",
                                       1:"1 vehicle",
                                       2:"2 vehicles",
                                       3:"3 vehicles",
                                       4:"4 or more vehicles"})


In [16]:
def stat_by_geog(df, geog):
    """
    Function to calculate statistics by geography and vehicle ownership.
    """
    
    # Group by equity geography and vehicle ownership
    df_grouped = df.groupby([geog, 'hhvehs'], as_index=False)['hhexpfac'].sum()
    
    # Calculate total households in each equity geography
    total_hh = df.groupby([geog], as_index=False)['hhexpfac'].sum().rename(columns={'hhexpfac': 'total_hh'})
    
    # Merge the grouped data with total households
    df_merged = df_grouped.merge(total_hh, on=geog)
    
    # Calculate percentage of households with the specified vehicle ownership
    df_merged['percentage'] = df_merged['hhexpfac'] / df_merged['total_hh']
    
    return df_merged.pivot(index=geog, columns='hhvehs', values='percentage')


In [17]:
df_region = df_hh.groupby(['hhvehs'], as_index=False)['hhexpfac'].sum()
df_region['percentage'] = df_region['hhexpfac'] / df_hh['hhexpfac'].sum()
df_region['hh_county'] = 'Region'

In [18]:
df = stat_by_geog(df_hh, 'hh_county')
pd.concat([df_region.pivot(index='hh_county', columns='hhvehs', values='percentage'),df])


hhvehs,0 vehicle,1 vehicle,2 vehicles,3 vehicles,4 or more vehicles
hh_county,,,,,
Region,7%,32%,37%,15%,8%
King,10%,35%,34%,13%,7%
Kitsap,4%,26%,39%,20%,11%
Pierce,5%,29%,39%,18%,10%
Snohomish,4%,27%,40%,18%,11%


In [19]:
df = stat_by_geog(df_hh, 'hh_rg_proposed')
df

hhvehs,0 vehicle,1 vehicle,2 vehicles,3 vehicles,4 or more vehicles
hh_rg_proposed,,,,,
Cities and Towns,2%,23%,43%,21%,11%
Core Cities,5%,34%,38%,15%,8%
High Capacity Transit Communities,4%,29%,40%,17%,10%
Metropolitan Cities,15%,42%,30%,9%,5%
Rural Areas,1%,16%,41%,27%,15%
Urban Unincorporated Areas,2%,21%,45%,21%,11%


In [20]:
df = stat_by_geog(df_hh, 'is_rgc')
df

hhvehs,0 vehicle,1 vehicle,2 vehicles,3 vehicles,4 or more vehicles
is_rgc,,,,,
In RGC,31%,49%,16%,3%,2%
Not in RGC,5%,30%,39%,17%,9%


In [21]:
df = stat_by_geog(df_hh, 'hh_rgc')
df

hhvehs,0 vehicle,1 vehicle,2 vehicles,3 vehicles,4 or more vehicles
hh_rgc,,,,,
Auburn,19%,48%,22%,7%,5%
Bellevue,22%,57%,17%,3%,1%
Bothell Canyon Park,2%,35%,41%,16%,7%
Bremerton,19%,50%,22%,6%,4%
Burien,14%,49%,25%,7%,4%
Everett,20%,53%,20%,5%,2%
Federal Way,15%,47%,25%,7%,6%
Greater Downtown Kirkland,10%,53%,28%,7%,2%
Kent,20%,47%,21%,8%,3%


In [22]:
df = pd.DataFrame()
for name, col in efa_names.items():
    _df = stat_by_geog(df_hh,col).reset_index()
    _df['Group'] = name
    _df.rename(columns={col: 'EFA Type'}, inplace=True)
    df = pd.concat([df, _df])

df.set_index(['Group','EFA Type'])

hhvehs                                             0 vehicle  1 vehicle  \
Group           EFA Type                                                  
People of Color Above Regional Average                    9%        34%   
                Below Regional Average                    6%        30%   
                Higher Share of Equity Population         9%        36%   
Income          Above Regional Average                    8%        34%   
                Below Regional Average                    6%        30%   
                Higher Share of Equity Population        13%        39%   
LEP             Above Regional Average                    7%        32%   
                Below Regional Average                    7%        31%   
                Higher Share of Equity Population         7%        36%   
Disability      Above Regional Average                    8%        33%   
                Below Regional Average                    6%        31%   
                Higher Share of Equity Population        11%        35%   
Older Adults    Above Regional Average                    6%        30%   
                Below Regional Average                    8%        34%   
                Higher Share of Equity Population         7%        30%   
Youth           Above Regional Average                    4%        27%   
                Below Regional Average                   11%        37%   
                Higher Share of Equity Population         3%        23%   

hhvehs                                             2 vehicles  3 vehicles  \
Group           EFA Type                                                    
People of Color Above Regional Average                    35%         14%   
                Below Regional Average                    38%         17%   
                Higher Share of Equity Population         34%         13%   
Income          Above Regional Average                    35%         14%   
                Below Regional Average                    39%         17%   
                Higher Share of Equity Population         31%         11%   
LEP             Above Regional Average                    37%         15%   
                Below Regional Average                    37%         16%   
                Higher Share of Equity Population         35%         13%   
Disability      Above Regional Average                    35%         15%   
                Below Regional Average                    39%         16%   
                Higher Share of Equity Population         32%         13%   
Older Adults    Above Regional Average                    38%         17%   
                Below Regional Average                    36%         14%   
                Higher Share of Equity Population         36%         18%   
Youth           Above Regional Average                    40%         19%   
                Below Regional Average                    33%         12%   
                Higher Share of Equity Population         43%         20%   

hhvehs                                             4 or more vehicles  
Group           EFA Type                                               
People of Color Above Regional Average                             8%  
                Below Regional Average                             9%  
                Higher Share of Equity Population                  7%  
Income          Above Regional Average                             8%  
                Below Regional Average                             9%  
                Higher Share of Equity Population                  6%  
LEP             Above Regional Average                             9%  
                Below Regional Average                             8%  
                Higher Share of Equity Population                  8%  
Disability      Above Regional Average                             9%  
                Below Regional Average                             9%  
                Higher